### DATA INGESTION


In [4]:
from langchain_core.documents import Document

In [5]:
doc = Document(page_content="This is Vedant learning RAG",
               metadat={
                   "source":"Example.txt",
                   "page":1,
                   "author":"Vedant"
               })

In [6]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python.txt")
documents = loader.load()

print(documents)

C:\Users\VEDANT DESHMUKH\AppData\Local\Temp\ipykernel_57004\1832807246.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


[Document(metadata={'source': '../data/text_files/python.txt'}, page_content='Python Basics 1. Python is a high-level programming language. 2. It uses\nindentation to define code blocks. 3. Variables can store different data\ntypes. 4. Lists are ordered and mutable collections. 5. Tuples are\nordered and immutable collections. 6. Dictionaries store data as\nkey-value pairs. 7. Functions are defined using the def keyword. 8.\nLoops include for and while statements. 9. Conditional logic uses if,\nelif, and else. 10. Modules help organize reusable code. 11. Exceptions\nare handled with try and except. 12. Python supports object-oriented\nprogramming.\n')]


In [7]:

from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

# 1. Define the path to your directory
path_to_pdfs = "../data/pdf"

# 2. Initialize the DirectoryLoader
# We use glob="**/*.pdf" to find all PDFs, including those in subfolders.
loader = DirectoryLoader(
    path_to_pdfs, 
    glob="**/*.pdf", 
    loader_cls=PyMuPDFLoader
)

# 3. Load the documents
docs = loader.load()
print(docs)
# Check how many documents/pages were loaded
print(f"Loaded {len(docs)} pages/documents.")

# Example: Print the content of the first page
if docs:
    print("\n--- First Page Content ---")
    print(docs[0].page_content[:500])  # Prints first 500 characters

[Document(metadata={'producer': 'xdvipdfmx (20220710)', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-09-10T17:29:00+00:00', 'source': '..\\data\\pdf\\IDCText  An Application for Conducting Text Input Research Studies in Indian Languages_CRC.pdf', 'file_path': '..\\data\\pdf\\IDCText  An Application for Conducting Text Input Research Studies in Indian Languages_CRC.pdf', 'total_pages': 21, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20240910172900-00'00'", 'page': 0}, page_content='IDCText : An Application for Conducting Text\nInput Research Studies in Indian\xa0Languages\nVedant Deshmukh1[0009−0000−2233−1634] and Anirudha\nJoshi2[0000−0002−1413−7329]\n1 Sardar Patel Institute of Technology (SPIT),Mumbai,India\nvedantdeshmukh3108@gmail.com\n2 Indian Institute of Technology (IIT), Bombay, Mumbai, India\nanirudha@iitb.ac.in\nAbstract Some might argue that text-input-related HCI re

In [8]:
type(docs[0])

langchain_core.documents.base.Document

In [9]:
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from pathlib import Path

# Fix: Update this specific import line
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [10]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: IDCText  An Application for Conducting Text Input Research Studies in Indian Languages_CRC.pdf
  ✓ Loaded 21 pages

Processing: Vedant_Deshmukh_Resume.pdf
  ✓ Loaded 1 pages

Total documents loaded: 22


In [11]:
all_pdf_documents

[Document(metadata={'producer': 'xdvipdfmx (20220710)', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-09-10T17:29:00+00:00', 'source': '..\\data\\pdf\\IDCText  An Application for Conducting Text Input Research Studies in Indian Languages_CRC.pdf', 'total_pages': 21, 'page': 0, 'page_label': '1', 'source_file': 'IDCText  An Application for Conducting Text Input Research Studies in Indian Languages_CRC.pdf', 'file_type': 'pdf'}, page_content='IDCText : An Application for Conducting Text\nInput Research Studies in Indian\xa0Languages\nVedant Deshmukh1[0009−0000−2233−1634] and Anirudha\nJoshi2[0000−0002−1413−7329]\n1 Sardar Patel Institute of Technology (SPIT),Mumbai,India\nvedantdeshmukh3108@gmail.com\n2 Indian Institute of Technology (IIT), Bombay, Mumbai, India\nanirudha@iitb.ac.in\nAbstract Some might argue that text-input-related HCI research in\nEnglish is verging towards saturation. On the other hand, a lot more\nwork remains to be done in Indian languages (and in fact, in

In [12]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """ Split documents into smaller chunks for better RAG performance """
    text_splitter= RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
         separators=["\n\n", "\n", " ", ""]
    )
    split_docs= text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [13]:
chunks=split_documents(all_pdf_documents)
chunks

Split 22 documents into 77 chunks

Example chunk:
Content: IDCText : An Application for Conducting Text
Input Research Studies in Indian Languages
Vedant Deshmukh1[0009−0000−2233−1634] and Anirudha
Joshi2[0000−0002−1413−7329]
1 Sardar Patel Institute of Techn...
Metadata: {'producer': 'xdvipdfmx (20220710)', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-09-10T17:29:00+00:00', 'source': '..\\data\\pdf\\IDCText  An Application for Conducting Text Input Research Studies in Indian Languages_CRC.pdf', 'total_pages': 21, 'page': 0, 'page_label': '1', 'source_file': 'IDCText  An Application for Conducting Text Input Research Studies in Indian Languages_CRC.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'xdvipdfmx (20220710)', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-09-10T17:29:00+00:00', 'source': '..\\data\\pdf\\IDCText  An Application for Conducting Text Input Research Studies in Indian Languages_CRC.pdf', 'total_pages': 21, 'page': 0, 'page_label': '1', 'source_file': 'IDCText  An Application for Conducting Text Input Research Studies in Indian Languages_CRC.pdf', 'file_type': 'pdf'}, page_content='IDCText : An Application for Conducting Text\nInput Research Studies in Indian\xa0Languages\nVedant Deshmukh1[0009−0000−2233−1634] and Anirudha\nJoshi2[0000−0002−1413−7329]\n1 Sardar Patel Institute of Technology (SPIT),Mumbai,India\nvedantdeshmukh3108@gmail.com\n2 Indian Institute of Technology (IIT), Bombay, Mumbai, India\nanirudha@iitb.ac.in\nAbstract Some might argue that text-input-related HCI research in\nEnglish is verging towards saturation. On the other hand, a lot more\nwork remains to be done in Indian languages (and in fact, in

In [14]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [15]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
         """Load the SentenceTransformer model"""
         try:
             print(f"Loading Embedding Model:{self.model_name}")
             self.model=SentenceTransformer(self.model_name)
             print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
         except:
             print(f"Error loading model {self.model_name}: {e}")
             raise

    def generate_embeddings(self,texts:List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
embedding_manager=EmbeddingManager()
embedding_manager

Loading Embedding Model:all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3550.32it/s]


Model loaded successfully. Embedding dimension: 384


# Setting Up Vector DB


In [16]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise        


vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 77


### Get data from chroma db

In [17]:
import chromadb

# Point this to the DIRECTORY containing the 'chroma.sqlite3' file (not the file itself)
client = chromadb.PersistentClient(path="../data/vector_store")

# List all collections inside the database
print("Collections found:", client.list_collections())

# Pick a collection to peek inside
# (Replace 'your_collection_name' with one of the names printed above)
collection = client.get_collection("pdf_documents")

# Fetch the data (include documents, metadatas, and optionally embeddings)
results = collection.get(
    include=["documents", "metadatas", "embeddings"]
)

# Print the first few items to see what's inside
print("\n--- SAMPLE DATA ---")
for i in range(min(5, len(results['ids']))):
    print(f"ID: {results['ids'][i]}")
    print(f"Document: {results['documents'][i]}")
    print(f"Metadata: {results['metadatas'][i]}")
    print(f"Embedding Vector (truncated): {results['embeddings'][i][:5]}...")
    print("-" * 20)

Collections found: [Collection(name=pdf_documents)]

--- SAMPLE DATA ---
ID: doc_03604b87_0
Document: IDCText : An Application for Conducting Text
Input Research Studies in Indian Languages
Vedant Deshmukh1[0009−0000−2233−1634] and Anirudha
Joshi2[0000−0002−1413−7329]
1 Sardar Patel Institute of Technology (SPIT),Mumbai,India
vedantdeshmukh3108@gmail.com
2 Indian Institute of Technology (IIT), Bombay, Mumbai, India
anirudha@iitb.ac.in
Abstract Some might argue that text-input-related HCI research in
English is verging towards saturation. On the other hand, a lot more
work remains to be done in Indian languages (and in fact, in several
other languages around the world). Through this paper, we release in
the public domain IDCText, a web based tool that makes it easier for re-
searchers to conduct text entry studies for Indian (and other) languages.
This application is compatible with all contemporary web browsers and
supports studies using any keyboard that can enter text in a web browse

In [18]:
texts = [docs.page_content for docs in chunks]


embeddings = embedding_manager.generate_embeddings(texts)
# Print the first 10 embeddings, showing just the first 5 numbers of each vector
print("--- TOP 10 EMBEDDINGS (TRUNCATED) ---")
for i, emb in enumerate(embeddings[:10]):
    print(f"Vector {i+1}: {emb[:5]}... (Total length: {len(emb)})")

vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 77 texts...


Batches: 100%|██████████| 3/3 [00:02<00:00,  1.14it/s]


Generated embeddings with shape: (77, 384)
--- TOP 10 EMBEDDINGS (TRUNCATED) ---
Vector 1: [-0.0804692  -0.0099913  -0.02882354 -0.02010746  0.00386067]... (Total length: 384)
Vector 2: [-0.05277626 -0.02072345 -0.01869433 -0.01040995 -0.0154268 ]... (Total length: 384)
Vector 3: [-0.04193973 -0.00024014  0.01306297 -0.04443382  0.01415739]... (Total length: 384)
Vector 4: [-0.01027254 -0.04573599  0.05831042 -0.04859436 -0.05727423]... (Total length: 384)
Vector 5: [ 0.03802326  0.0038115   0.07110264 -0.05430204 -0.05521917]... (Total length: 384)
Vector 6: [ 0.0588237  -0.06072084 -0.01684852 -0.04104782 -0.1075073 ]... (Total length: 384)
Vector 7: [-0.00804828 -0.04773644 -0.00624943 -0.03167697 -0.10225508]... (Total length: 384)
Vector 8: [-0.0664588   0.0554772  -0.06666445 -0.05126511 -0.0191479 ]... (Total length: 384)
Vector 9: [-0.04107703  0.03968127 -0.01161955 -0.01007422 -0.06431872]... (Total length: 384)
Vector 10: [-0.06050269  0.00363591 -0.06511912 -0.00825453 -0.0

In [19]:

class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        




In [20]:
rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [21]:
rag_retriever.retrieve("what is error rates ?" )

Retrieving documents for query: 'what is error rates ?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 12.40it/s]

Generated embeddings with shape: (1, 384)
Retrieved 4 documents (after filtering)


[{'id': 'doc_76a75d3b_17',
  'content': 'scription study, but it is considered to be valid only if the error rates are within\nreasonable limits (e.g. 5% on average). So before reporting speed, the HCI\nresearcher is also interested in reporting errors made by participants.\nSoukoreff et al. define several standard metrics for calculating errors in a\ntranscription study [ 25]. The first is the Minimum String Distance (MSD) Error\nRate, as shown in equation 3. Here, C stands for “correct” characters that are\ncommon between the presented phrase and transcribed phrase and INF stands\nfor “incorrect but not fixed” characters in the transcribed phrase including extra\ncharacters (insertions), incorrect characters (substitutions), and omitted char-\nacters (deletions).\nMSD Error rate = INF\nC + IN F × 100% (3)\nPlease note that, MSD error rate is based only on the finally transcribed\nphrase and does not include incorrect characters that were typed by the par-\nticipant but were later del

In [ ]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")
 

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.3-70b-versatile",
    temperature=0.1,
    max_tokens=1024
)
## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    ## retrieve the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answer using GROQ LLM
    # Notice we keep the f-string here, it populates context and query immediately!
    prompt = f"""Use the following context to answer the question in bit detail And is the product good ? .
    
    Context:
    {context}
    
    Question: {query}
    
    Answer:"""
    
    # FIX: Pass 'prompt' directly inside the list without calling .format()
    response = llm.invoke([prompt])
    return response.content

In [ ]:
answer=rag_simple("what is the IDCText paper about ?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is IDCText about ?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  8.54it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


IDCText is an application designed to conduct text input studies in Indian languages. It allows researchers to set up and manage studies, including specifying the design of the study (between-subjects or within-subjects), and provides features such as user management for longitudinal studies. The application also includes special features for text input studies in Indian languages, such as accounting for Unicode exceptions and variations, and a ramped study design to help first-time text inputters.

The application seems to be a valuable tool for researchers, as it provides features that are not available in current tools, such as the ability to choose between a within-subjects or a between-subjects study, and the option to conduct longitudinal studies. Additionally, IDCText's support for Unicode and its ability to handle complexities of text input in Indian languages make it a useful tool for studying text input in these languages.

Overall, IDCText appears to be a well-designed and u